# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
files = os.listdir('/kaggle/input/noisy-drone-rf-signal-classification-v2/drone_RF_data')
print(files[0])
print("Total files:", len(files))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/noisy-drone-rf-signal-classification-v2/drone_RF_data'

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)

In [ ]:
import os
base = '/kaggle/input/datasets/sgluege/noisy-drone-rf-signal-classification-v2'
print(os.listdir(base))

In [ ]:
base = '/kaggle/input/datasets/sgluege/noisy-drone-rf-signal-classification-v2/drone_RF_data'
files = os.listdir(base)
print(files[0])
print("Total files:", len(files))

In [ ]:
import torch

sample_path = '/kaggle/input/datasets/sgluege/noisy-drone-rf-signal-classification-v2/drone_RF_data/IQdata_sample884_target0_snr-12.pt'
data = torch.load(sample_path)

print(type(data))
print(data)

In [ ]:
print("x_iq shape:", data['x_iq'].shape)
print("y (label):", data['y'].item())
print("snr:", data['snr'].item())

In [ ]:
import re

files = os.listdir(base)
targets = set()
for f in files:
    match = re.search(r'target(\d+)', f)
    if match:
        targets.add(int(match.group(1)))

print("Unique classes:", sorted(targets))
print("Number of classes:", len(targets))

In [ ]:
from collections import Counter

class_counts = Counter()
for f in files:
    match = re.search(r'target(\d+)', f)
    if match:
        class_counts[int(match.group(1))] += 1

for cls in sorted(class_counts):
    print(f"Class {cls}: {class_counts[cls]} samples")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def iq_to_spectrogram(x_iq, n_fft=256, hop_length=128):
    # x_iq shape: [2, N] -> combine I and Q into complex signal
    i_channel = x_iq[0]
    q_channel = x_iq[1]
    complex_signal = torch.complex(i_channel, q_channel)
    
    # STFT
    spec = torch.stft(complex_signal, n_fft=n_fft, hop_length=hop_length, 
                       window=torch.hann_window(n_fft), return_complex=True)
    
    # Convert to magnitude (log scale, since RF signals have huge dynamic range)
    magnitude = torch.abs(spec)
    log_magnitude = torch.log1p(magnitude)
    
    return log_magnitude

# Test on our sample
spec = iq_to_spectrogram(data['x_iq'])
print("Spectrogram shape:", spec.shape)

plt.figure(figsize=(8, 6))
plt.imshow(spec.numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.title(f"Spectrogram - Class {data['y'].item()}, SNR {data['snr'].item()}")
plt.xlabel("Time")
plt.ylabel("Frequency")
plt.colorbar(label='Log Magnitude')
plt.show()

In [ ]:
print("Spectrogram shape:", spec.shape)

In [2]:
import os
import re
from sklearn.model_selection import train_test_split

base = '/kaggle/input/datasets/sgluege/noisy-drone-rf-signal-classification-v2/drone_RF_data'
all_files = os.listdir(base)

# Extract labels for stratification
labels = []
for f in all_files:
    match = re.search(r'target(\d+)', f)
    labels.append(int(match.group(1)))

# Split: 70% train, 15% val, 15% test — stratified by class
train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files, labels, test_size=0.3, stratify=labels, random_state=42
)
val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))

AttributeError: 'NoneType' object has no attribute 'group'

In [ ]:
bad_files = []
for f in all_files:
    match = re.search(r'target(\d+)', f)
    if match is None:
        bad_files.append(f)

print("Number of bad files:", len(bad_files))
print(bad_files[:10])  # show first 10

In [ ]:
# Filter out non-data files
all_files = [f for f in all_files if f.endswith('.pt')]

# Extract labels for stratification
labels = []
for f in all_files:
    match = re.search(r'target(\d+)', f)
    labels.append(int(match.group(1)))

print("Total valid files:", len(all_files))

In [ ]:
from sklearn.model_selection import train_test_split

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files, labels, test_size=0.3, stratify=labels, random_state=42
)
val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))

In [ ]:
import time

start = time.time()
batch_x, batch_y = next(iter(train_loader))
end = time.time()

print("Batch shape:", batch_x.shape)
print("Labels shape:", batch_y.shape)
print("Time to load one batch:", end - start, "seconds")

In [ ]:
import torch
from torch.utils.data import Dataset
import os

class DroneRFDataset(Dataset):
    def __init__(self, file_list, base_dir, n_fft=256, hop_length=128, target_size=(128, 128)):
        self.file_list = file_list
        self.base_dir = base_dir
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.target_size = target_size

    def __len__(self):
        return len(self.file_list)

    def iq_to_spectrogram(self, x_iq):
        i_channel = x_iq[0]
        q_channel = x_iq[1]
        complex_signal = torch.complex(i_channel, q_channel)
        spec = torch.stft(complex_signal, n_fft=self.n_fft, hop_length=self.hop_length,
                           window=torch.hann_window(self.n_fft), return_complex=True)
        magnitude = torch.abs(spec)
        log_magnitude = torch.log1p(magnitude)
        log_magnitude = log_magnitude.unsqueeze(0).unsqueeze(0)
        resized = torch.nn.functional.interpolate(log_magnitude, size=self.target_size, mode='bilinear')
        return resized.squeeze(0)

    def __getitem__(self, idx):
        file_path = os.path.join(self.base_dir, self.file_list[idx])
        data = torch.load(file_path)
        spec = self.iq_to_spectrogram(data['x_iq'])
        label = data['y']
        return spec, label

In [ ]:
from torch.utils.data import DataLoader

train_dataset = DroneRFDataset(train_files, base, target_size=(128, 128))
val_dataset = DroneRFDataset(val_files, base, target_size=(128, 128))
test_dataset = DroneRFDataset(test_files, base, target_size=(128, 128))

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("Data loaders created successfully")

In [ ]:
import time

start = time.time()
batch_x, batch_y = next(iter(train_loader))
end = time.time()

print("Batch shape:", batch_x.shape)
print("Labels shape:", batch_y.shape)
print("Time to load one batch:", end - start, "seconds")

In [ ]:
import torch
import os
from tqdm import tqdm

cache_dir = '/kaggle/working/spectrogram_cache'
os.makedirs(cache_dir, exist_ok=True)

def precompute_spectrograms(file_list, base_dir, cache_dir, n_fft=256, hop_length=128, target_size=(128, 128)):
    for fname in tqdm(file_list):
        cache_path = os.path.join(cache_dir, fname.replace('.pt', '_spec.pt'))
        if os.path.exists(cache_path):
            continue  # already cached, skip
        
        data = torch.load(os.path.join(base_dir, fname))
        x_iq = data['x_iq']
        
        i_channel = x_iq[0]
        q_channel = x_iq[1]
        complex_signal = torch.complex(i_channel, q_channel)
        spec = torch.stft(complex_signal, n_fft=n_fft, hop_length=hop_length,
                           window=torch.hann_window(n_fft), return_complex=True)
        magnitude = torch.abs(spec)
        log_magnitude = torch.log1p(magnitude)
        log_magnitude = log_magnitude.unsqueeze(0).unsqueeze(0)
        resized = torch.nn.functional.interpolate(log_magnitude, size=target_size, mode='bilinear')
        resized = resized.squeeze(0)
        
        torch.save({'spec': resized, 'label': data['y']}, cache_path)

# Precompute for all splits
precompute_spectrograms(train_files, base, cache_dir)
precompute_spectrograms(val_files, base, cache_dir)
precompute_spectrograms(test_files, base, cache_dir)

In [ ]:
class CachedSpectrogramDataset(Dataset):
    def __init__(self, file_list, cache_dir):
        self.file_list = file_list
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]
        cache_path = os.path.join(self.cache_dir, fname.replace('.pt', '_spec.pt'))
        cached = torch.load(cache_path)
        return cached['spec'], cached['label']

In [ ]:
train_dataset = CachedSpectrogramDataset(train_files, cache_dir)
val_dataset = CachedSpectrogramDataset(val_files, cache_dir)
test_dataset = CachedSpectrogramDataset(test_files, cache_dir)

batch_size = 32  # can increase now since loading is much cheaper
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("Fast data loaders created")

In [ ]:
import time

start = time.time()
batch_x, batch_y = next(iter(train_loader))
end = time.time()

print("Batch shape:", batch_x.shape)
print("Labels shape:", batch_y.shape)
print("Time to load one batch:", end - start, "seconds")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DroneRFClassifier(nn.Module):
    def __init__(self, num_classes=7):
        super(DroneRFClassifier, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        
        self.pool = nn.MaxPool2d(2, 2)
        
        # After 3 pooling layers: 128 -> 64 -> 32 -> 16
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        x = x.view(x.size(0), -1)  # flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = DroneRFClassifier(num_classes=7)
print(model)

In [ ]:
import torch
from collections import Counter

# Compute class weights based on training set distribution
class_counts_train = Counter(train_labels)
total_train = len(train_labels)
num_classes = 7

class_weights = []
for c in range(num_classes):
    weight = total_train / (num_classes * class_counts_train[c])
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights:", class_weights)

In [ ]:


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

model = model.to(device)
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import re
from collections import Counter
from sklearn.model_selection import train_test_split

In [ ]:
base = '/kaggle/input/datasets/sgluege/noisy-drone-rf-signal-classification-v2/drone_RF_data'
all_files = os.listdir(base)
all_files = [f for f in all_files if f.endswith('.pt')]

labels = []
for f in all_files:
    match = re.search(r'target(\d+)', f)
    labels.append(int(match.group(1)))

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files, labels, test_size=0.3, stratify=labels, random_state=42
)
val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print("Train:", len(train_files), "Val:", len(val_files), "Test:", len(test_files))

In [ ]:
cache_dir = '/kaggle/working/spectrogram_cache'

class CachedSpectrogramDataset(Dataset):
    def __init__(self, file_list, cache_dir):
        self.file_list = file_list
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]
        cache_path = os.path.join(self.cache_dir, fname.replace('.pt', '_spec.pt'))
        cached = torch.load(cache_path)
        return cached['spec'], cached['label']

In [ ]:
train_dataset = CachedSpectrogramDataset(train_files, cache_dir)
val_dataset = CachedSpectrogramDataset(val_files, cache_dir)
test_dataset = CachedSpectrogramDataset(test_files, cache_dir)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("Data loaders ready")

In [ ]:
class DroneRFClassifier(nn.Module):
    def __init__(self, num_classes=7):
        super(DroneRFClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = DroneRFClassifier(num_classes=7)

In [ ]:
class_counts_train = Counter(train_labels)
total_train = len(train_labels)
num_classes = 7

class_weights = []
for c in range(num_classes):
    weight = total_train / (num_classes * class_counts_train[c])
    class_weights.append(weight)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

model = model.to(device)
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            running_loss += loss.item() * batch_x.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
class DroneRFDataset(Dataset):
    def __init__(self, file_list, base_dir, n_fft=256, hop_length=128, target_size=(128, 128)):
        self.file_list = file_list
        self.base_dir = base_dir
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.target_size = target_size

    def __len__(self):
        return len(self.file_list)

    def iq_to_spectrogram(self, x_iq):
        i_channel = x_iq[0]
        q_channel = x_iq[1]
        complex_signal = torch.complex(i_channel, q_channel)
        spec = torch.stft(complex_signal, n_fft=self.n_fft, hop_length=self.hop_length,
                           window=torch.hann_window(self.n_fft), return_complex=True)
        magnitude = torch.abs(spec)
        log_magnitude = torch.log1p(magnitude)
        log_magnitude = log_magnitude.unsqueeze(0).unsqueeze(0)
        resized = torch.nn.functional.interpolate(log_magnitude, size=self.target_size, mode='bilinear')
        return resized.squeeze(0)

    def __getitem__(self, idx):
        file_path = os.path.join(self.base_dir, self.file_list[idx])
        data = torch.load(file_path)
        spec = self.iq_to_spectrogram(data['x_iq'])
        label = data['y']
        return spec, label

In [ ]:
from tqdm import tqdm

cache_dir = '/kaggle/working/spectrogram_cache'
os.makedirs(cache_dir, exist_ok=True)

def precompute_spectrograms(file_list, base_dir, cache_dir, n_fft=256, hop_length=128, target_size=(128, 128)):
    for fname in tqdm(file_list):
        cache_path = os.path.join(cache_dir, fname.replace('.pt', '_spec.pt'))
        if os.path.exists(cache_path):
            continue
        
        data = torch.load(os.path.join(base_dir, fname))
        x_iq = data['x_iq']
        
        i_channel = x_iq[0]
        q_channel = x_iq[1]
        complex_signal = torch.complex(i_channel, q_channel)
        spec = torch.stft(complex_signal, n_fft=n_fft, hop_length=hop_length,
                           window=torch.hann_window(n_fft), return_complex=True)
        magnitude = torch.abs(spec)
        log_magnitude = torch.log1p(magnitude)
        log_magnitude = log_magnitude.unsqueeze(0).unsqueeze(0)
        resized = torch.nn.functional.interpolate(log_magnitude, size=target_size, mode='bilinear')
        resized = resized.squeeze(0)
        
        torch.save({'spec': resized, 'label': data['y']}, cache_path)

precompute_spectrograms(train_files, base, cache_dir)
precompute_spectrograms(val_files, base, cache_dir)
precompute_spectrograms(test_files, base, cache_dir)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import re
from collections import Counter
from sklearn.model_selection import train_test_split

In [ ]:
cache_dir = '/kaggle/working/spectrogram_cache'

class CachedSpectrogramDataset(Dataset):
    def __init__(self, file_list, cache_dir):
        self.file_list = file_list
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]
        cache_path = os.path.join(self.cache_dir, fname.replace('.pt', '_spec.pt'))
        cached = torch.load(cache_path)
        return cached['spec'], cached['label']

In [ ]:
train_dataset = CachedSpectrogramDataset(train_files, cache_dir)
val_dataset = CachedSpectrogramDataset(val_files, cache_dir)
test_dataset = CachedSpectrogramDataset(test_files, cache_dir)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("Data loaders ready")

In [ ]:
x, y = next(iter(train_loader))
print("Input shape:", x.shape)
print("Label shape:", y.shape)
print("Unique labels in batch:", y.unique())
print("Input min/max:", x.min().item(), x.max().item())

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

class DroneCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 16),    # 128 -> 64
            conv_block(16, 32),   # 64 -> 32
            conv_block(32, 64),   # 32 -> 16
            conv_block(64, 128),  # 16 -> 8
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

model = DroneCNN(num_classes=7).to(device)

# Test with one real batch
out = model(x.to(device))
print("Output shape:", out.shape)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
import torch.nn.functional as F

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)

            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, correct / total

print("Training setup ready")

In [ ]:
import time

epochs = 10
best_val_acc = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f} | "
          f"{time.time()-start:.0f}s")

print("Best val accuracy:", best_val_acc)

In [ ]:
import torch.nn.functional as F

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)

            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, correct / total

print("Training setup ready")

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pth", map_location=device))
test_loss, test_acc = evaluate(model, test_loader)
print(f"Test loss {test_loss:.4f} | Test acc {test_acc:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb.to(device))
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(yb.tolist())

print(classification_report(all_labels, all_preds, digits=3))
print(confusion_matrix(all_labels, all_preds))

In [ ]:
import re
from collections import defaultdict

snrs = [int(re.search(r'snr(-?\d+)', str(f)).group(1)) for f in test_files]
print("Files:", len(snrs), "| Predictions:", len(all_preds))

stats = defaultdict(lambda: [0, 0])
for s, p, l in zip(snrs, all_preds, all_labels):
    stats[s][1] += 1
    stats[s][0] += int(p == l)

for s in sorted(stats):
    c, t = stats[s]
    print(f"SNR {s:>4} dB: acc {c/t:.3f} ({t} samples)")

In [ ]:
import re
import numpy as np

train_targets = [int(re.search(r'target(\d+)', str(f)).group(1)) for f in train_files]
counts = np.bincount(train_targets, minlength=7)
print("Train class counts:", counts)

weights = counts.sum() / (7 * counts)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

model = DroneCNN(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print("Fresh model ready")

In [ ]:
import time

epochs = 10
best_val_loss = float("inf")
history_w = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    history_w["train_loss"].append(train_loss)
    history_w["train_acc"].append(train_acc)
    history_w["val_loss"].append(val_loss)
    history_w["val_acc"].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/kaggle/working/best_model_weighted.pth")

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f} | "
          f"{time.time()-start:.0f}s")

print("Best val loss:", best_val_loss)

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model_weighted.pth", map_location=device))

epochs2 = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs2)
best_val_loss = float("inf")

for epoch in range(epochs2):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/kaggle/working/best_model_weighted2.pth")

    print(f"Epoch {epoch+1}/{epochs2} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f} | "
          f"lr {scheduler.get_last_lr()[0]:.5f} | {time.time()-start:.0f}s")

print("Best val loss:", best_val_loss)

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model_weighted2.pth", map_location=device))
model.eval()

all_preds2, all_labels2 = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb.to(device))
        all_preds2.extend(out.argmax(1).cpu().tolist())
        all_labels2.extend(yb.tolist())

print(classification_report(all_labels2, all_preds2, digits=3))
print(confusion_matrix(all_labels2, all_preds2))

In [ ]:
soft_weights = np.sqrt(weights)
class_weights = torch.tensor(soft_weights, dtype=torch.float32).to(device)
print("Soft weights:", class_weights)

model = DroneCNN(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs3 = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs3)
best_val_loss = float("inf")

for epoch in range(epochs3):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/kaggle/working/best_model_soft.pth")

    print(f"Epoch {epoch+1}/{epochs3} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f} | {time.time()-start:.0f}s")

print("Best val loss:", best_val_loss)

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model_soft.pth", map_location=device))
model.eval()

all_preds3, all_labels3 = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb.to(device))
        all_preds3.extend(out.argmax(1).cpu().tolist())
        all_labels3.extend(yb.tolist())

print(classification_report(all_labels3, all_preds3, digits=3))
print(confusion_matrix(all_labels3, all_preds3))

In [ ]:
stats3 = defaultdict(lambda: [0, 0])
for s, p, l in zip(snrs, all_preds3, all_labels3):
    stats3[s][1] += 1
    stats3[s][0] += int(p == l)

for s in sorted(stats3):
    c, t = stats3[s]
    print(f"SNR {s:>4} dB: acc {c/t:.3f}")

In [ ]:
print(classification_report(all_labels4, all_preds4, digits=3))
print(confusion_matrix(all_labels4, all_preds4))

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model_aug.pth", map_location=device))
model.eval()

all_preds4, all_labels4 = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb.to(device))
        all_preds4.extend(out.argmax(1).cpu().tolist())
        all_labels4.extend(yb.tolist())

print(classification_report(all_labels4, all_preds4, digits=3))
print(confusion_matrix(all_labels4, all_preds4))

In [ ]:
d = torch.load(raw_files[0])
x = d["x_iq"]                                  # [2, 1048576]
power = (x[0]**2 + x[1]**2)                    # power per sample
chunks = power.view(64, -1).mean(1)            # 64 equal time chunks
print("File:", raw_files[0].split("/")[-1])
print("Label:", int(d["y"]), "| SNR:", int(d["snr"]))
print((chunks / chunks.mean()).numpy().round(2))

In [ ]:
f = next(p for p in raw_files if "target0_snr20" in p)
d = torch.load(f)
x = d["x_iq"]
power = (x[0]**2 + x[1]**2)
chunks = power.view(64, -1).mean(1)
print("File:", f.split("/")[-1])
print((chunks / chunks.mean()).numpy().round(2))

In [ ]:
import torch.nn.functional as F
N = 65536
n = len(raw_files)
X = np.lib.format.open_memmap("/kaggle/working/raw_crop2.npy", mode="w+",
                              dtype=np.float16, shape=(n, 2, N))
starts = np.zeros(n, dtype=np.int64)

start_t = time.time()
for i, f in enumerate(raw_files):
    d = torch.load(f)
    x = d["x_iq"]
    p = (x[0]**2 + x[1]**2).double()
    cs = F.pad(p.cumsum(0), (1, 0))
    w = cs[N:] - cs[:-N]              # power inside every window
    s = int(w.argmax())
    starts[i] = s
    c = x[:, s:s + N]
    c = c / (c.std() + 1e-8)
    X[i] = c.numpy().astype(np.float16)
    if i % 2000 == 0:
        print(i, f"{time.time()-start_t:.0f}s")

X.flush()
np.save("/kaggle/working/raw_starts.npy", starts)
print("Done in", f"{(time.time()-start_t)/60:.1f} min")

In [3]:
import torch.nn.functional as F
N = 65536
n = len(raw_files)
X = np.lib.format.open_memmap("/kaggle/working/raw_crop2.npy", mode="w+",
                              dtype=np.float16, shape=(n, 2, N))
starts = np.zeros(n, dtype=np.int64)

start_t = time.time()
for i, f in enumerate(raw_files):
    d = torch.load(f)
    x = d["x_iq"]
    p = (x[0]**2 + x[1]**2).double()
    cs = F.pad(p.cumsum(0), (1, 0))
    w = cs[N:] - cs[:-N]              # power inside every window
    s = int(w.argmax())
    starts[i] = s
    c = x[:, s:s + N]
    c = c / (c.std() + 1e-8)
    X[i] = c.numpy().astype(np.float16)
    if i % 2000 == 0:
        print(i, f"{time.time()-start_t:.0f}s")

X.flush()
np.save("/kaggle/working/raw_starts.npy", starts)
print("Done in", f"{(time.time()-start_t)/60:.1f} min")

NameError: name 'raw_files' is not defined

In [4]:
import os, time, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

print(sorted(os.listdir("/kaggle/working")))

with open("/kaggle/working/raw_files.txt") as fh:
    raw_files = fh.read().split("\n")
print("raw_files:", len(raw_files))
print(raw_files[0])

Using: cuda
['.virtual_documents']


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/raw_files.txt'

In [5]:
import os
for name in sorted(os.listdir("/kaggle/working")):
    p = os.path.join("/kaggle/working", name)
    size = os.path.getsize(p) / 1e6 if os.path.isfile(p) else 0
    print(name, f"{size:.1f} MB" if size else "(folder)")

.virtual_documents (folder)
